# Project 1 - Superstore Sales - Real Upwork Client Task
**Client Brief:** "My Shopify export is messy. Clean it and tell me which categories lose money when we discount >20%."
**Dataset:** 2.2M Superstore real - 9994 rows

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('superstore_real.csv')
print(f"Shape: {df.shape}")
df.head()

Shape: (10800, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164


In [18]:
# Real client file audit - 
print("Missing values:\n", df.isnull().sum().sum(), "total missing")
print("\nDuplicates:", df.duplicated().sum())
print("\nDiscount > 0.2 losing money?")
print(df[df['Discount']>0.2]['Profit'].describe())

# Your first client insight
df['Profitable'] = df['Profit'] > 0
loss_rate = 1 - df['Profitable'].mean()
print(f"\n{loss_rate:.1%} of orders lose money!")

Missing values:
 15325 total missing

Duplicates: 504

Discount > 0.2 losing money?
count    1393.000000
mean      -97.183098
std       327.542304
min     -6599.978000
25%       -71.994000
50%       -18.252500
75%        -6.023700
max      1119.996800
Name: Profit, dtype: float64

25.4% of orders lose money!


In [19]:
# Which category bleeds money on high discuonts? 
high_disc = df[df['Discount'] > 0.2]

result = high_disc.groupby('Category').agg(
    orders = ('Order ID', 'count'),
    avg_discount = ('Discount', 'mean'),
    total_profit = ('Profit', 'sum'),
    loss_orders = ('Profit', lambda x: (x<0).sum())
).sort_values('total_profit')

print(result)
print("\nWorst Sub-Category: ")
print(high_disc.groupby('Sub-Category')['Profit'].sum().sort_values().head(5))

                 orders  avg_discount  total_profit  loss_orders
Category                                                        
Furniture           542      0.425258   -54477.7561          527
Office Supplies     680      0.744118   -47140.1376          680
Technology          171      0.444444   -33758.1623          141

Worst Sub-Category: 
Sub-Category
Binders      -38510.4964
Tables       -30698.2228
Machines     -29555.3521
Bookcases    -11097.7614
Appliances    -8629.6412
Name: Profit, dtype: float64


In [22]:
# Clean and save for portfolio
df_clean = df.drop_duplicates()
df_clean.to_csv('superstore_cleaned.csv', index=False)

# Create portfolio summary
summary = f"""
# Project 1 Result - E-commerce Loss Analysis
- Raw rows: {len(df)} | Clean rows: {len(df_clean)}
- Loss rate: 25.4%
- Worst Category on high discount: Furniture (${result.loc['Furniture','total_profit']:.0f})
- Worst Sub-Categories: Binders, Tables, Machines
- Client Action: Stop discounting Tables >20% - losing $30k
"""

with open('README.md','w') as f:
    f.write(summary)

print(summary)
print("\nSaved: superstore_cleaned.csv + README.md")


# Project 1 Result - E-commerce Loss Analysis
- Raw rows: 10800 | Clean rows: 10296
- Loss rate: 25.4%
- Worst Category on high discount: Furniture ($-54478)
- Worst Sub-Categories: Binders, Tables, Machines
- Client Action: Stop discounting Tables >20% - losing $30k


Saved: superstore_cleaned.csv + README.md
